# Extract and stitch clean 32x32 validation windows

Slides a **32x32 window** (configurable) across every validation tile,
checks each window for NaN/Inf, non-binary mask, implausible elevation, or
void `dem_bic` values, and **rebuilds one stitched cleaned tile per source**.

This version **does not save the individual 32×32 patches**. It only writes
the final stitched output where accepted windows are placed back into their
original positions and rejected windows stay as `NaN`.

Reads both `.npy` and `.npz` source tiles (same tolerant loader used
elsewhere). Writes plain `.npy` stitched tiles, so the output plugs straight
into downstream code that expects one tensor per tile.

Set `DRY_RUN = True` (default) to see counts before writing anything.


In [1]:
import os
from pathlib import Path
from collections import Counter

import numpy as np
from tqdm.auto import tqdm


In [2]:
def load_hma_npy(path):
    """Load one HMA tensor file (.npy or .npz), tolerant of:

    Legacy .npy: a plain (4, H, W) float array -- dem_bic, lidar_raw, mask, gt_dem.

    Extended .npy: the same 4 channels plus per-file georeferencing metadata
    appended after them --
        Ch5: GDAL geotransform, shape (6,) float64, (x0, dx, 0, y0, 0, -dy)
        Ch6: EPSG code, single int.
    Since these extra fields don't share the (H, W) shape of the image
    channels, they're packed as a heterogeneous container (object-dtype
    array wrapping a dict, or a list/tuple of 6 items).

    .npz: an NpzFile of named arrays -- either 4 separate channel arrays
    (dem_bic/lidar_raw/mask/gt_dem, or Ch1..Ch4) plus optional geotransform/
    epsg, or a single combined (4,H,W) array under a common key name.

    Returns (image, geotransform, epsg):
        image        -- (4, H, W) float32 ndarray
        geotransform -- (6,) float64 ndarray, or None
        epsg         -- int, or None
    """
    path_str = str(path)

    if path_str.endswith('.npz'):
        with np.load(path, allow_pickle=True) as npz:
            obj = {k: npz[k] for k in npz.files}
    else:
        try:
            raw = np.load(path, allow_pickle=False)
        except ValueError:
            raw = np.load(path, allow_pickle=True)

        if raw.dtype != object and raw.ndim == 3 and raw.shape[0] == 4:
            return raw.astype(np.float32, copy=False), None, None

        obj = raw
        if isinstance(raw, np.ndarray) and raw.dtype == object:
            obj = raw.item() if raw.shape == () else raw

    if isinstance(obj, dict):
        def _get(*names):
            for n in names:
                if n in obj:
                    return obj[n]
            return None

        combined = _get("data", "image", "stack", "tensor", "arr_0")
        if combined is not None and np.asarray(combined).ndim == 3 and np.asarray(combined).shape[0] == 4:
            combined = np.asarray(combined, dtype=np.float32)
            dem_bic, lidar_raw, mask, gt_dem = combined[0], combined[1], combined[2], combined[3]
        else:
            dem_bic   = _get("dem_bic", "Ch1", "ch1")
            lidar_raw = _get("lidar_raw", "Ch2", "ch2")
            mask      = _get("mask", "Ch3", "ch3")
            gt_dem    = _get("gt_dem", "Ch4", "ch4")

        geotransform = _get("geotransform", "gt", "affine", "Ch5", "ch5")
        epsg         = _get("epsg", "EPSG", "Ch6", "ch6")
    elif isinstance(obj, (list, tuple, np.ndarray)) and len(obj) >= 4:
        dem_bic, lidar_raw, mask, gt_dem = obj[0], obj[1], obj[2], obj[3]
        geotransform = obj[4] if len(obj) > 4 else None
        epsg = obj[5] if len(obj) > 5 else None
    else:
        raise ValueError(
            "Unrecognized container for " + str(path) + ": type=" + str(type(obj))
        )

    if dem_bic is None or lidar_raw is None or mask is None or gt_dem is None:
        raise ValueError(
            "Could not find all 4 channels in " + str(path) + ". Keys: " +
            str(list(obj.keys()) if isinstance(obj, dict) else "n/a")
        )

    image = np.stack([
        np.asarray(dem_bic, dtype=np.float32),
        np.asarray(lidar_raw, dtype=np.float32),
        np.asarray(mask, dtype=np.float32),
        np.asarray(gt_dem, dtype=np.float32),
    ], axis=0)
    geotransform = np.asarray(geotransform, dtype=np.float64) if geotransform is not None else None
    epsg = int(epsg) if epsg is not None else None
    return image, geotransform, epsg


In [3]:
# ============================================================================
# CONFIG
# ============================================================================
# SOURCE_VAL_DIR = Path(r'D:\Vaibhav\dem-lidar\dataset_12m\tensors\validation_contiguous')
# DEST_VAL_DIR   = Path(r'D:\Vaibhav\dem-lidar\dataset_12m\tensors\validation_clean_32')
SOURCE_VAL_DIR = Path(r'D:\Vaibhav\dem-lidar\tensors_256\all')
DEST_VAL_DIR   = Path(r'D:\Vaibhav\dem-lidar\tensors_256')

WINDOW = 32   # patch size
STRIDE = 32   # 32 = non-overlapping tiling; smaller = overlapping (more patches, some redundant)

# Bad-window criteria (a window is dropped if ANY are true)
CHECK_NAN_INF        = True
CHECK_MASK_BINARY    = True
MASK_TOLERANCE       = 1e-6
CHECK_ELEV_PLAUSIBLE = True
ELEV_MIN_PLAUSIBLE   = -500.0
ELEV_MAX_PLAUSIBLE   = 9000.0
CHECK_DEM_VOID       = True
DEM_MIN_VOID_THRESHOLD = 50.0   # dem_bic.min() in the window below this -> void/no-data suspected

PRESERVE_GEOTRANSFORM = True   # if the source tile has one, shift it per-window and keep it

DRY_RUN = False   # True: report only, writes nothing. Flip to False once counts look right.

DEST_VAL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Source: {SOURCE_VAL_DIR}')
print(f'Dest:   {DEST_VAL_DIR}')
print(f'Window: {WINDOW}x{WINDOW}, stride {STRIDE}')
print(f'DRY_RUN = {DRY_RUN}')


Source: D:\Vaibhav\dem-lidar\tensors_256\all
Dest:   D:\Vaibhav\dem-lidar\tensors_256
Window: 32x32, stride 32
DRY_RUN = False


In [4]:
def bad_window_reasons(dem_bic, lidar_raw, mask, gt_dem):
    reasons = []

    if CHECK_NAN_INF:
        for name, arr in [('dem_bic', dem_bic), ('lidar_raw', lidar_raw), ('mask', mask), ('gt_dem', gt_dem)]:
            if not np.all(np.isfinite(arr)):
                reasons.append(f'nan_inf_{name}')

    if CHECK_MASK_BINARY and np.all(np.isfinite(mask)):
        is_binary = np.all((np.abs(mask) < MASK_TOLERANCE) | (np.abs(mask - 1.0) < MASK_TOLERANCE))
        if not is_binary:
            reasons.append('non_binary_mask')

    if CHECK_ELEV_PLAUSIBLE:
        for name, arr in [('dem_bic', dem_bic), ('gt_dem', gt_dem)]:
            finite = arr[np.isfinite(arr)]
            if finite.size > 0 and (finite.min() < ELEV_MIN_PLAUSIBLE or finite.max() > ELEV_MAX_PLAUSIBLE):
                reasons.append(f'implausible_elevation_{name}')

    if CHECK_DEM_VOID:
        finite = dem_bic[np.isfinite(dem_bic)]
        if finite.size > 0 and finite.min() < DEM_MIN_VOID_THRESHOLD:
            reasons.append('dem_bic_void_suspected')

    return reasons


def shift_geotransform(geotransform, row_off, col_off):
    """Translate a GDAL geotransform (x0, dx, 0, y0, 0, -dy) by (col_off, row_off) pixels."""
    if geotransform is None:
        return None
    x0, dx, rot1, y0, rot2, neg_dy = geotransform
    new_x0 = x0 + col_off * dx + row_off * rot1
    new_y0 = y0 + col_off * rot2 + row_off * neg_dy
    return np.array([new_x0, dx, rot1, new_y0, rot2, neg_dy], dtype=np.float64)


## Save a stitched cleaned tile

This rebuilds one output tile per source image by placing each accepted
32×32 window back into its original position.

Rejected windows are left as `NaN`, so the stitched output shows exactly
which parts of the source tile survived the quality checks.


In [5]:
SAVE_STITCHED_ORIGINAL = True
STITCH_FILL_VALUE = np.nan

def save_stitched_original(image, kept_windows, src_path, dst_root, geotransform=None, epsg=None):
    '''Save a stitched tile where accepted windows are written back into place.'''
    stitched = np.full(image.shape, STITCH_FILL_VALUE, dtype=np.float32)

    for row, col, window in kept_windows:
        stitched[:, row:row + WINDOW, col:col + WINDOW] = window.astype(np.float32)

    out_dir = dst_root / "stitched_originals" / src_path.parent.relative_to(SOURCE_VAL_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{src_path.stem}_stitched.npy"

    if geotransform is not None or epsg is not None:
        payload = {
            "dem_bic": stitched[0],
            "lidar_raw": stitched[1],
            "mask": stitched[2],
            "gt_dem": stitched[3],
        }
        if geotransform is not None:
            payload["geotransform"] = geotransform
        if epsg is not None:
            payload["epsg"] = epsg
        np.save(out_path, np.array(payload, dtype=object), allow_pickle=True)
    else:
        np.save(out_path, stitched)

    return out_path


In [6]:
# ============================================================================
# WALK SOURCE, SLIDE WINDOW, WRITE ONLY THE STITCHED CLEAN TILES
# ============================================================================
filepaths = []
for root, _, files in os.walk(SOURCE_VAL_DIR):
    for f in files:
        if f.endswith('.npy') or f.endswith('.npz'):
            filepaths.append(Path(root) / f)

print(f'Found {len(filepaths)} source validation tiles.')

n_windows_total = 0
n_windows_kept = 0
reason_counts = Counter()
per_file_kept = []   # (path, n_kept, n_total) for the summary
stitched_outputs = []

for path in tqdm(filepaths, desc='Extracting clean patches'):
    rel_dir = path.parent.relative_to(SOURCE_VAL_DIR)
    stem = path.stem

    try:
        image, geotransform, epsg = load_hma_npy(path)
    except Exception as e:
        tqdm.write(f'SKIPPING (load error): {path} -> {e}')
        continue

    _, h, w = image.shape
    file_kept = 0
    file_total = 0
    kept_windows = []

    for row in range(0, h - WINDOW + 1, STRIDE):
        for col in range(0, w - WINDOW + 1, STRIDE):
            file_total += 1
            n_windows_total += 1

            window = image[:, row:row + WINDOW, col:col + WINDOW]
            dem_bic, lidar_raw, mask, gt_dem = window[0], window[1], window[2], window[3]

            reasons = bad_window_reasons(dem_bic, lidar_raw, mask, gt_dem)
            if reasons:
                for r in reasons:
                    reason_counts[r] += 1
                continue

            file_kept += 1
            n_windows_kept += 1
            kept_windows.append((row, col, window))

    per_file_kept.append((str(path), file_kept, file_total))

    if SAVE_STITCHED_ORIGINAL:
        if STRIDE != WINDOW:
            tqdm.write(f'WARNING: stitching is exact only when STRIDE == WINDOW. Current STRIDE={STRIDE}, WINDOW={WINDOW}.')
        if not DRY_RUN:
            stitched_path = save_stitched_original(
                image=image,
                kept_windows=kept_windows,
                src_path=path,
                dst_root=DEST_VAL_DIR,
                geotransform=geotransform,
                epsg=epsg,
            )
            stitched_outputs.append((str(path), str(stitched_path), file_kept, file_total))

print(f'\nWindows examined: {n_windows_total}')
print(f'Windows kept:     {n_windows_kept} ({n_windows_kept/max(n_windows_total,1)*100:.1f}%)')
print('\nDrop reasons:')
for reason, count in reason_counts.most_common():
    print(f'  {reason:30s} {count}')


Found 70 source validation tiles.


Extracting clean patches:   0%|          | 0/70 [00:00<?, ?it/s]


Windows examined: 396645
Windows kept:     361650 (91.2%)

Drop reasons:
  dem_bic_void_suspected         34995


In [7]:
# ============================================================================
# PER-FILE BREAKDOWN + WHAT TO DO NEXT
# ============================================================================
zero_kept = [p for p, kept, total in per_file_kept if kept == 0 and total > 0]
print(f'{len(zero_kept)} / {len(per_file_kept)} source tiles yielded ZERO clean windows.')
if zero_kept:
    print('Worst offenders (entirely bad):')
    for p in zero_kept[:10]:
        print(f'  {p}')

if DRY_RUN:
    print('\nDRY_RUN is True -- nothing written. Review the counts above, set DRY_RUN = False, and re-run this cell + the one above.')
else:
    print(f'\nClean stitched tiles written to: {DEST_VAL_DIR}')
    print('\nEach output file is one stitched mosaic built from the accepted 32×32 windows.')
    print('Rejected regions remain NaN.')
    print('\nNEXT STEP -- update your fine-tune notebook:')
    print(f"  VAL_DIRS = [r'{DEST_VAL_DIR}']")
    print('  Make sure the downstream loader expects one stitched tile per file.')


0 / 70 source tiles yielded ZERO clean windows.

Clean stitched tiles written to: D:\Vaibhav\dem-lidar\tensors_256

Each output file is one stitched mosaic built from the accepted 32×32 windows.
Rejected regions remain NaN.

NEXT STEP -- update your fine-tune notebook:
  VAL_DIRS = [r'D:\Vaibhav\dem-lidar\tensors_256']
  Make sure the downstream loader expects one stitched tile per file.
